### INGESTA DE DATOS A LA CAPA RAW
**IMPORTAMOS LAS LIBRERIAS**

In [0]:
import requests
import base64
import time
from datetime import datetime
import json
from pathlib import Path

**CONEXIÓN CON LA API DE SPOTIFY**

In [0]:
url_token = "https://accounts.spotify.com/api/token"
client_id = dbutils.secrets.get(scope="api-keys", key="spotify-api-id")
client_secret = dbutils.secrets.get(scope="api-keys", key="spotify-api-key")

response = requests.post(
    url_token,
    data={
        "grant_type": "client_credentials"
    },
    auth=(
        client_id,
        client_secret
    )
)

token = response.json()["access_token"]
print("Token obtenido correctamente")

In [0]:
def spotify_get(base_url, endpoint, params=None):
    url = f"{base_url}{endpoint}"

    headers = {
        "Authorization": f"Bearer {token}"
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code == 429:
        retry_after = int(
            response.headers.get("Retry-After", "5")
        )
        print(f"Rate limit. Esperando {retry_after} segundos...")
        time.sleep(retry_after)

        return spotify_get(endpoint, params)

    response.raise_for_status()
    return response.json()

In [0]:
def search_tracks(query, limit=10, offset=0):
    base_url = "https://api.spotify.com/v1"

    return spotify_get(base_url,"/search",
        params={
            "q": query,
            "type": "track",
            "limit": limit,
            "offset": offset
        }
    )

In [0]:
dbutils.widgets.multiselect(
    "generos",
    "rock",
    ["rock", "pop", "latin", "reggaeton", "hip-hop", "rap", "jazz", "classical",   "electronic", "metal", "indie", "alternative", "country", "r&b", "salsa",    "bachata", "merengue"],
    "Generos"
)

dbutils.widgets.dropdown(
    "limite",
    "10",
    ["1", "5", "10"],
    "Limite"
)

In [0]:
raw_records = []

limite = int(dbutils.widgets.get("limite"))
generos = dbutils.widgets.get("generos").split(",")

for genero in generos:

    for offset in range(0, 100, 10):

        data = search_tracks(
            query=genero,
            limit=limite,
            offset=offset
        )

        raw_records.append({
            "search_term": genero,
            "offset": offset,
            "extraction_timestamp": datetime.utcnow(),
            "payload": json.dumps(data)
        })

print("Conexión correcta")

**ALMACENAR LOS DATOS EN LA CAPA RAW**

In [0]:
raw_df = spark.createDataFrame(raw_records)

display(raw_df)

In [0]:
print("Cantidad de Registros en Capa Raw: ", raw_df.count())

In [0]:
current_date = datetime.utcnow().date()
raw_dir = "proyecto_spotify.raw.spotify_raw_"+str(current_date.year) + "_" + str(current_date.month) + "_" + str(current_date.day) 

(
    raw_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(
        raw_dir
    )
)